# ChatGPT Conversation History - Overview & Statistics

This notebook provides a comprehensive overview of your ChatGPT conversation history.

**Data Source:** chatgpt_backup.json (1,221 conversations, 23,497 messages)

---

## Setup and Data Loading

In [7]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [8]:
# Load the parsed ChatGPT data
import os
if not os.path.exists('conversations.json'):
    raise FileNotFoundError("conversations.json not found — put it in the repo root or update the path.")
with open('conversations.json', 'r', encoding='utf-8') as f:
    conversations = json.load(f)

# Helper to robustly access and count messages in a conversation
def safe_messages(conv):
    """Return a list of message dicts for a conversation (never None)."""
    if not isinstance(conv, dict):
        return []
    msgs = conv.get('messages', [])
    # messages may be a JSON string — try to parse
    if isinstance(msgs, str):
        try:
            msgs = json.loads(msgs)
        except Exception:
            return []
    # nested structure sometimes stores messages under a wrapper
    if isinstance(msgs, dict) and 'messages' in msgs:
        msgs = msgs['messages']
    if isinstance(msgs, (list, tuple)):
        return msgs
    return []

def safe_count(conv):
    # Prefer explicit field if present and valid
    if not isinstance(conv, dict):
        return 0
    if 'message_count' in conv and conv.get('message_count') is not None:
        try:
            return int(conv['message_count'])
        except Exception:
            pass
    return len(safe_messages(conv))

# Populate missing `message_count` in-place so downstream cells don't KeyError

for c in conversations:
    try:
        if 'message_count' not in c or c.get('message_count') is None:
            c['message_count'] = safe_count(c)
    except Exception:
        c['message_count'] = 0

print(f"Loaded {len(conversations):,} conversations")
print(f"Total messages: {sum(c['message_count'] for c in conversations):,}")

Loaded 1,221 conversations
Total messages: 0


## 1. High-Level Statistics

In [9]:
# Calculate key statistics
total_conversations = len(conversations)
total_messages = sum(c['message_count'] for c in conversations)
avg_messages_per_conv = total_messages / total_conversations

# Count messages by role
role_counts = Counter()
for conv in conversations:
    for msg in safe_messages(conv):
        role_counts[msg.get('role', 'unknown')] += 1

# Date range
create_dates = [datetime.fromisoformat(c['create_date']) for c in conversations if c['create_date']]
update_dates = [datetime.fromisoformat(c['update_date']) for c in conversations if c['update_date']]

earliest = min(create_dates)
latest = max(update_dates)
duration_days = (latest - earliest).days

print("=" * 60)
print("CHATGPT CONVERSATION HISTORY OVERVIEW")
print("=" * 60)
print(f"\n📊 OVERALL STATISTICS")
print(f"   Total Conversations: {total_conversations:,}")
print(f"   Total Messages: {total_messages:,}")
print(f"   Average Messages/Conversation: {avg_messages_per_conv:.1f}")
print(f"\n📅 TIME RANGE")
print(f"   Earliest: {earliest.strftime('%Y-%m-%d')}")
print(f"   Latest: {latest.strftime('%Y-%m-%d')}")
print(f"   Duration: {duration_days} days (~{duration_days/365:.1f} years)")
print(f"   Avg Conversations/Day: {total_conversations/duration_days:.2f}")
print(f"\n💬 MESSAGE BREAKDOWN")
for role, count in role_counts.most_common():
    pct = (count / total_messages) * 100
    print(f"   {role.capitalize()}: {count:,} ({pct:.1f}%)")
print("=" * 60)

KeyError: 'messages'

## 2. Conversation Length Distribution

In [10]:
# Get message counts
message_counts = [c['message_count'] for c in conversations]

# Statistics
print("CONVERSATION LENGTH STATISTICS")
print(f"Mean: {np.mean(message_counts):.1f} messages")
print(f"Median: {np.median(message_counts):.0f} messages")
print(f"Std Dev: {np.std(message_counts):.1f}")
print(f"Min: {min(message_counts)} messages")
print(f"Max: {max(message_counts)} messages")
print(f"\nPercentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    val = np.percentile(message_counts, p)
    print(f"  {p}th: {val:.0f} messages")

CONVERSATION LENGTH STATISTICS
Mean: 0.0 messages
Median: 0 messages
Std Dev: 0.0
Min: 0 messages
Max: 0 messages

Percentiles:
  25th: 0 messages
  50th: 0 messages
  75th: 0 messages
  90th: 0 messages
  95th: 0 messages
  99th: 0 messages


In [ ]:
# Visualize distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram
axes[0, 0].hist(message_counts, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(np.mean(message_counts), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(message_counts):.1f}')
axes[0, 0].axvline(np.median(message_counts), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(message_counts):.0f}')
axes[0, 0].set_xlabel('Messages per Conversation')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Conversation Lengths')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Box plot
axes[0, 1].boxplot(message_counts, vert=True)
axes[0, 1].set_ylabel('Messages per Conversation')
axes[0, 1].set_title('Conversation Length Box Plot')
axes[0, 1].grid(True, alpha=0.3)

# Cumulative distribution
sorted_counts = np.sort(message_counts)
cumulative = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts) * 100
axes[1, 0].plot(sorted_counts, cumulative, linewidth=2)
axes[1, 0].set_xlabel('Messages per Conversation')
axes[1, 0].set_ylabel('Cumulative Percentage')
axes[1, 0].set_title('Cumulative Distribution')
axes[1, 0].grid(True, alpha=0.3)

# Category breakdown
categories = {
    'Quick (1-5)': len([c for c in message_counts if c <= 5]),
    'Short (6-10)': len([c for c in message_counts if 6 <= c <= 10]),
    'Medium (11-20)': len([c for c in message_counts if 11 <= c <= 20]),
    'Long (21-40)': len([c for c in message_counts if 21 <= c <= 40]),
    'Very Long (41+)': len([c for c in message_counts if c > 40])
}
axes[1, 1].bar(categories.keys(), categories.values(), edgecolor='black')
axes[1, 1].set_xlabel('Conversation Category')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Conversations by Length Category')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print category breakdown
print("\nCONVERSATION LENGTH CATEGORIES:")
for cat, count in categories.items():
    pct = (count / total_conversations) * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")

## 3. Top 20 Longest Conversations

In [ ]:
# Find longest conversations
sorted_convs = sorted(conversations, key=lambda x: x['message_count'], reverse=True)

print("TOP 20 LONGEST CONVERSATIONS")
print("=" * 100)
print(f"{'Rank':<6} {'Msgs':<6} {'Date':<12} {'Title':<75}")
print("=" * 100)

for i, conv in enumerate(sorted_convs[:20], 1):
    title = conv['title'][:70] + '...' if len(conv['title']) > 70 else conv['title']
    date = conv['create_date'][:10] if conv['create_date'] else 'Unknown'
    print(f"{i:<6} {conv['message_count']:<6} {date:<12} {title}")

print("=" * 100)

In [ ]:
# Visualize top 20
top_20 = sorted_convs[:20]
titles = [c['title'][:40] + '...' if len(c['title']) > 40 else c['title'] for c in top_20]
msg_counts = [c['message_count'] for c in top_20]

plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(titles)), msg_counts, edgecolor='black')

# Color gradient
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(bars)))
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.yticks(range(len(titles)), titles)
plt.xlabel('Number of Messages')
plt.title('Top 20 Longest Conversations', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 4. Message Role Analysis

In [ ]:
# Detailed role analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
roles = list(role_counts.keys())
counts = list(role_counts.values())
colors_pie = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']

axes[0].pie(counts, labels=roles, autopct='%1.1f%%', startangle=90, colors=colors_pie[:len(roles)])
axes[0].set_title('Message Distribution by Role', fontweight='bold')

# Bar chart
axes[1].bar(roles, counts, edgecolor='black', color=colors_pie[:len(roles)])
axes[1].set_ylabel('Message Count')
axes[1].set_title('Messages by Role', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (role, count) in enumerate(zip(roles, counts)):
    axes[1].text(i, count, f'{count:,}\n({count/total_messages*100:.1f}%)', 
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate user-to-assistant ratio
user_msgs = role_counts.get('user', 0)
assistant_msgs = role_counts.get('assistant', 0)
ratio = assistant_msgs / user_msgs if user_msgs > 0 else 0

print(f"\nUSER-ASSISTANT INTERACTION ANALYSIS")
print("=" * 50)
print(f"User Messages: {user_msgs:,}")
print(f"Assistant Messages: {assistant_msgs:,}")
print(f"Ratio (Assistant/User): {ratio:.2f}")
print(f"\nInterpretation:")
if ratio > 1.5:
    print("  • High ratio suggests complex questions requiring")
    print("    detailed, multi-part responses")
elif ratio > 1.0:
    print("  • Balanced conversation with follow-up clarifications")
else:
    print("  • Quick Q&A style interactions")

if role_counts.get('tool', 0) > 0:
    tool_msgs = role_counts['tool']
    print(f"\nTool Messages: {tool_msgs:,}")
    print(f"  • Indicates {tool_msgs} tool/function calls")
    print(f"  • Heavy use of advanced features like:")
    print(f"    - Code execution")
    print(f"    - Web browsing")
    print(f"    - DALL-E image generation")
    print(f"    - File operations")

## 5. Model Usage Analysis

In [ ]:
# Extract model information
model_counts = Counter()
for conv in conversations:
    for msg in safe_messages(conv):
        if msg.get('role') == 'assistant' and msg.get('model'):
            model_counts[msg['model']] += 1

print("MODEL USAGE BREAKDOWN")
print("=" * 60)
total_model_msgs = sum(model_counts.values())
for model, count in model_counts.most_common():
    pct = (count / total_model_msgs) * 100 if total_model_msgs > 0 else 0
    print(f"{model:<30} {count:>6,} messages ({pct:>5.1f}%)")
print("=" * 60)

In [ ]:
# Visualize model usage
if len(model_counts) > 0:
    models = list(model_counts.keys())
    counts = list(model_counts.values())
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(models)), counts, edgecolor='black')
    
    # Color bars
    colors = plt.cm.Set3(np.linspace(0, 1, len(bars)))
    for bar, color in zip(bars, colors):
        bar.set_color(color)
    
    plt.xticks(range(len(models)), models, rotation=45, ha='right')
    plt.ylabel('Message Count')
    plt.title('Messages by Model', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, count in enumerate(counts):
        plt.text(i, count, f'{count:,}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("No model information available in the data.")

## 6. Create Summary DataFrame

In [ ]:
# Create a comprehensive DataFrame for further analysis
df_data = []
for conv in conversations:
    # Count messages by role in this conversation
    msgs = safe_messages(conv)
    conv_user_msgs = sum(1 for m in msgs if m.get('role') == 'user')
    conv_assistant_msgs = sum(1 for m in msgs if m.get('role') == 'assistant')
    conv_tool_msgs = sum(1 for m in msgs if m.get('role') == 'tool')
    
    # Extract date
    create_date = datetime.fromisoformat(conv['create_date']) if conv['create_date'] else None
    
    df_data.append({
        'index': conv.get('index'),
        'title': conv.get('title'),
        'create_date': create_date,
        'message_count': conv.get('message_count', len(msgs)),
        'user_messages': conv_user_msgs,
        'assistant_messages': conv_assistant_msgs,
        'tool_messages': conv_tool_msgs,
        'ratio': conv_assistant_msgs / conv_user_msgs if conv_user_msgs > 0 else 0
    })

df = pd.DataFrame(df_data)
df = df.sort_values('create_date', ascending=False).reset_index(drop=True)

print(f"Created DataFrame with {len(df)} conversations\n")
print("Sample of data:")
display(df.head(10))

print("\nDataFrame Statistics:")
display(df.describe())

In [ ]:
# Save DataFrame for use in other notebooks
df.to_csv('../chatgpt_conversations_dataframe.csv', index=False)
print("✓ DataFrame saved to 'chatgpt_conversations_dataframe.csv'")

## 7. Key Insights Summary

In [ ]:
print("\n" + "=" * 70)
print("KEY INSIGHTS FROM YOUR CHATGPT CONVERSATION HISTORY")
print("=" * 70)

print(f"\n📊 VOLUME")
print(f"   • You've had {total_conversations:,} conversations over {duration_days} days")
print(f"   • That's {total_conversations/duration_days:.1f} conversations per day on average")
print(f"   • Total of {total_messages:,} messages exchanged")

print(f"\n💬 CONVERSATION STYLE")
print(f"   • Average conversation: {avg_messages_per_conv:.1f} messages")
print(f"   • Median conversation: {np.median(message_counts):.0f} messages")
if avg_messages_per_conv > 15:
    print(f"   • Your conversations are in-depth and exploratory")
elif avg_messages_per_conv > 8:
    print(f"   • Balanced mix of quick queries and detailed discussions")
else:
    print(f"   • Primarily quick Q&A style interactions")

print(f"\n🎯 INTERACTION PATTERN")
print(f"   • Assistant-to-User ratio: {ratio:.2f}")
if ratio > 1.2:
    print(f"   • Questions often require detailed, multi-part answers")
print(f"   • Tool usage: {role_counts.get('tool', 0):,} tool messages")
if role_counts.get('tool', 0) > 1000:
    print(f"   • Heavy user of advanced features (code execution, browsing, etc.)")

print(f"\n🔥 MOST ACTIVE TOPICS (by conversation length)")
for i, conv in enumerate(sorted_convs[:5], 1):
    print(f"   {i}. {conv['title'][:60]} ({conv['message_count']} msgs)")

print(f"\n📅 TEMPORAL PATTERN")
# Calculate conversations per month
df['year_month'] = df['create_date'].dt.to_period('M')
monthly_counts = df['year_month'].value_counts().sort_index()
avg_per_month = monthly_counts.mean()
print(f"   • Average {avg_per_month:.1f} conversations per month")
print(f"   • Most active month: {monthly_counts.idxmax()} ({monthly_counts.max()} conversations)")
print(f"   • Least active month: {monthly_counts.idxmin()} ({monthly_counts.min()} conversations)")

print("\n" + "=" * 70)
print("Next: Run notebook 02 for temporal analysis and trends over time")
print("=" * 70)